# Conway's Game of Life - Parallel with OpenMP

## The Rules (just 4)

Each cell is either **alive** (1) or **dead** (0). Every generation, each cell looks at its **8 neighbours**:

```
┌───┬───┬───┐
│ N │ N │ N │
├───┼───┼───┤
│ N │ X │ N │   X = current cell, N = neighbours
├───┼───┼───┤
│ N │ N │ N │
└───┴───┴───┘
```

| Cell state | Neighbours | Next state | Rule name |
|---|---|---|---|
| Alive | < 2 | Dies | Underpopulation |
| Alive | 2 or 3 | Survives | Survival |
| Alive | > 3 | Dies | Overpopulation |
| Dead | exactly 3 | Born | Reproduction |

Simple rules → incredibly complex emergent behaviour.

---

## The Parallelism Problem

Every cell needs to **read** its neighbours and **write** its new state simultaneously.
If we read and write the same grid, a cell updated early would corrupt the neighbours of cells updated later.

The solution: **double buffering** - two grids, always read from one, write to the other, then swap.

---
## 1. Setup & Grid Utilities

In [1]:
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define ROWS     8
#define COLS     16
#define THREADS  4

// Two grids - the core of double buffering
// We ALWAYS read from grid[0], write to grid[1], then swap pointers
int grid_A[ROWS][COLS];
int grid_B[ROWS][COLS];
int (*current)[COLS] = grid_A;   // pointer to the grid we READ from
int (*next)[COLS]    = grid_B;   // pointer to the grid we WRITE to

// Print the grid nicely
void print_grid(int g[ROWS][COLS], int gen) {
    printf("Generation %d:\n", gen);
    for (int r = 0; r < ROWS; r++) {
        printf("|");
        for (int c = 0; c < COLS; c++)
            printf("%c", g[r][c] ? '#' : '.');
        printf("|\n");
    }
}

// Count living cells
int count_alive(int g[ROWS][COLS]) {
    int count = 0;
    for (int r = 0; r < ROWS; r++)
        for (int c = 0; c < COLS; c++)
            count += g[r][c];
    return count;
}

printf("Grid size: %d x %d = %d cells\n", ROWS, COLS, ROWS * COLS);
printf("Threads: %d\n", THREADS);
printf("Each thread handles ~%d rows\n", ROWS / THREADS);

Grid size: 8 x 16 = 128 cells
Threads: 4
Each thread handles ~2 rows


---
## 2. Why Double Buffering?

Before we parallelize, let's understand the core problem.

```
WRONG - single grid (read and write same grid):

    Step 1: update cell (2,3) → writes new value into grid[2][3]
    Step 2: update cell (2,4) → reads grid[2][3] as a neighbour
        BUT grid[2][3] is already the NEW value!
        Cell (2,4) is computing based on corrupted data.

CORRECT - double buffer:
    
    current[][] = old state (READ ONLY)
    next[][]    = new state (WRITE ONLY)

    All cells read from current[][] simultaneously - no corruption possible
    All cells write to next[][] simultaneously - no conflicts
    After all cells done: swap pointers (next becomes current)
```

This is exactly what makes it safe to parallelize - threads can compute any cell in any order because they all read from the same frozen snapshot.

---
## 3. The Step Function (Core Logic)

`collapse(2)` flattens both loops so OpenMP splits all **128 cells** (8×16) across 4 threads — 32 cells per thread.

In [2]:
// Count live neighbours of cell (r, c)
// Uses wrap-around edges (toroidal grid — top connects to bottom, left to right)
int count_neighbours(int g[ROWS][COLS], int r, int c) {
    int count = 0;
    for (int dr = -1; dr <= 1; dr++) {
        for (int dc = -1; dc <= 1; dc++) {
            if (dr == 0 && dc == 0) continue;  // skip self
            int nr = (r + dr + ROWS) % ROWS;   // wrap around vertically
            int nc = (c + dc + COLS) % COLS;   // wrap around horizontally
            count += g[nr][nc];
        }
    }
    return count;
}

// One generation step - parallel with OpenMP
// READ from: src (frozen snapshot)
// WRITE to:  dst (fresh empty grid)
void step(int src[ROWS][COLS], int dst[ROWS][COLS]) {
    #pragma omp parallel for num_threads(THREADS) collapse(2)
    for (int r = 0; r < ROWS; r++) {
        for (int c = 0; c < COLS; c++) {
            int neighbours = count_neighbours(src, r, c);
            int alive = src[r][c];

            // Apply the 4 rules
            if (alive && neighbours < 2)           dst[r][c] = 0;  // underpopulation
            else if (alive && neighbours <= 3)     dst[r][c] = 1;  // survival
            else if (alive && neighbours > 3)      dst[r][c] = 0;  // overpopulation
            else if (!alive && neighbours == 3)    dst[r][c] = 1;  // reproduction
            else                                   dst[r][c] = 0;  // stays dead
        }
    }
}

printf("step() and count_neighbours() defined.\n");
printf("collapse(2) flattens the 2D loop into 1D so OpenMP can split %d*%d=%d cells across %d threads\n",
       ROWS, COLS, ROWS*COLS, THREADS);

step() and count_neighbours() defined.
collapse(2) flattens the 2D loop into 1D so OpenMP can split 8*16=128 cells across 4 threads


---
## 4. `collapse(2)` Explained

Without `collapse(2)`, OpenMP only splits the **outer loop** (rows):
```
Thread 0 → rows 0-1   (each doing all 16 columns)
Thread 1 → rows 2-3
Thread 2 → rows 4-5
Thread 3 → rows 6-7
```
With `collapse(2)`, OpenMP merges both loops and splits **all 128 cells**:
```
Thread 0 → cells 0-31
Thread 1 → cells 32-63
Thread 2 → cells 64-95
Thread 3 → cells 96-127
```
On a small 8×16 grid this matters more — without collapse, each thread only gets 2 rows which is very coarse granularity.

---
## 5. Pattern 1: Blinker (Period-2 Oscillator)

The simplest oscillator - 3 cells in a line that flip between horizontal and vertical every generation. Placed in the middle of the 8×16 grid at row 4, col 8.

In [3]:
void run_blinker() {
    // Reset both grids
    memset(grid_A, 0, sizeof(grid_A));
    memset(grid_B, 0, sizeof(grid_B));
    current = grid_A;
    next    = grid_B;

    // Place a horizontal blinker in the middle
    int r = ROWS / 2;
    int c = COLS / 2;
    current[r][c-1] = 1;
    current[r][c]   = 1;
    current[r][c+1] = 1;

    printf("=== Blinker (should alternate horizontal/vertical) ===\n\n");
    print_grid(current, 0);

    for (int gen = 1; gen <= 4; gen++) {
        step(current, next);

        // Swap buffers — next becomes current, current becomes next
        int (*tmp)[COLS] = current;
        current = next;
        next    = tmp;

        print_grid(current, gen);
    }
}

run_blinker();

=== Blinker (should alternate horizontal/vertical) ===

Generation 0:
|................|
|................|
|................|
|................|
|.......###......|
|................|
|................|
|................|
Generation 1:
|................|
|................|
|................|
|........#.......|
|........#.......|
|........#.......|
|................|
|................|
Generation 2:
|................|
|................|
|................|
|................|
|.......###......|
|................|
|................|
|................|
Generation 3:
|................|
|................|
|................|
|........#.......|
|........#.......|
|........#.......|
|................|
|................|
Generation 4:
|................|
|................|
|................|
|................|
|.......###......|
|................|
|................|
|................|


---
## 6. Pattern 2: Glider

The glider moves diagonally across the grid forever - it takes 4 generations to return to its original shape but shifted by 1 cell diagonally.

In [4]:
void run_glider() {
    memset(grid_A, 0, sizeof(grid_A));
    memset(grid_B, 0, sizeof(grid_B));
    current = grid_A;
    next    = grid_B;

    // Classic glider pattern:
    //  .X.
    //  ..X
    //  XXX
    current[1][2] = 1;
    current[2][3] = 1;
    current[3][1] = 1;
    current[3][2] = 1;
    current[3][3] = 1;

    printf("=== Glider (watch it move diagonally) ===\n\n");
    print_grid(current, 0);

    for (int gen = 1; gen <= 8; gen++) {
        step(current, next);
        int (*tmp)[COLS] = current;
        current = next;
        next    = tmp;

        //if (gen % 2 == 0)   // print every 2 generations to keep output short
            print_grid(current, gen);
    }
}

run_glider();

=== Glider (watch it move diagonally) ===

Generation 0:
|................|
|..#.............|
|...#............|
|.###............|
|................|
|................|
|................|
|................|
Generation 1:
|................|
|................|
|.#.#............|
|..##............|
|..#.............|
|................|
|................|
|................|
Generation 2:
|................|
|................|
|...#............|
|.#.#............|
|..##............|
|................|
|................|
|................|
Generation 3:
|................|
|................|
|..#.............|
|...##...........|
|..##............|
|................|
|................|
|................|
Generation 4:
|................|
|................|
|...#............|
|....#...........|
|..###...........|
|................|
|................|
|................|
Generation 5:
|................|
|................|
|................|
|..#.#...........|
|...##...........|
|...#............|

---
## 7. Performance: Sequential vs Parallel

In [5]:
#define BENCH_ROWS  512
#define BENCH_COLS  512
#define GENERATIONS 100

// Allocate large grids on the heap for benchmarking
int *bench_A = (int*)calloc(BENCH_ROWS * BENCH_COLS, sizeof(int));
int *bench_B = (int*)calloc(BENCH_ROWS * BENCH_COLS, sizeof(int));

#define BENCH_IDX(r, c) ((r) * BENCH_COLS + (c))

// Seed with random pattern
srand(42);
for (int i = 0; i < BENCH_ROWS * BENCH_COLS; i++)
    bench_A[i] = rand() % 2;

// Sequential step
void step_seq(int *src, int *dst) {
    for (int r = 0; r < BENCH_ROWS; r++) {
        for (int c = 0; c < BENCH_COLS; c++) {
            int n = 0;
            for (int dr = -1; dr <= 1; dr++)
                for (int dc = -1; dc <= 1; dc++) {
                    if (dr == 0 && dc == 0) continue;
                    int nr = (r + dr + BENCH_ROWS) % BENCH_ROWS;
                    int nc = (c + dc + BENCH_COLS) % BENCH_COLS;
                    n += src[BENCH_IDX(nr, nc)];
                }
            int alive = src[BENCH_IDX(r, c)];
            dst[BENCH_IDX(r, c)] = (alive && n == 2) || n == 3 ? 1 : 0;
        }
    }
}

// Parallel step
void step_par(int *src, int *dst) {
    #pragma omp parallel for num_threads(THREADS) collapse(2)
    for (int r = 0; r < BENCH_ROWS; r++) {
        for (int c = 0; c < BENCH_COLS; c++) {
            int n = 0;
            for (int dr = -1; dr <= 1; dr++)
                for (int dc = -1; dc <= 1; dc++) {
                    if (dr == 0 && dc == 0) continue;
                    int nr = (r + dr + BENCH_ROWS) % BENCH_ROWS;
                    int nc = (c + dc + BENCH_COLS) % BENCH_COLS;
                    n += src[BENCH_IDX(nr, nc)];
                }
            int alive = src[BENCH_IDX(r, c)];
            dst[BENCH_IDX(r, c)] = (alive && n == 2) || n == 3 ? 1 : 0;
        }
    }
}

// Benchmark sequential
int *src = bench_A, *dst = bench_B;
double t0 = omp_get_wtime();
for (int g = 0; g < GENERATIONS; g++) {
    step_seq(src, dst);
    int *tmp = src; src = dst; dst = tmp;
}
double t_seq = omp_get_wtime() - t0;

// Reset and benchmark parallel
srand(42);
for (int i = 0; i < BENCH_ROWS * BENCH_COLS; i++) bench_A[i] = rand() % 2;
src = bench_A; dst = bench_B;
t0 = omp_get_wtime();
for (int g = 0; g < GENERATIONS; g++) {
    step_par(src, dst);
    int *tmp = src; src = dst; dst = tmp;
}
double t_par = omp_get_wtime() - t0;

printf("Grid: %dx%d = %d cells, %d generations\n",
       BENCH_ROWS, BENCH_COLS, BENCH_ROWS * BENCH_COLS, GENERATIONS);
printf("=============================================\n");
printf("  Sequential:  %.4f s\n", t_seq);
printf("  Parallel:    %.4f s  (%.1fx speedup)\n", t_par, t_seq / t_par);
printf("=============================================\n");

free(bench_A);
free(bench_B);

Grid: 512x512 = 262144 cells, 100 generations
  Sequential:  1.3265 s
  Parallel:    0.3410 s  (3.9x speedup)


---
## 8. Why Double Buffering Has Zero Race Conditions

Let's prove to ourselves it's safe by thinking through what each thread touches:

```
Generation N (frozen - nobody writes here):
┌─────────────────────────────────────┐
│  current[][]  READ ONLY             │
│  Thread 0 reads rows 0-1            │
│  Thread 1 reads rows 2-3            │
│  Thread 2 reads rows 4-5            │
│  Thread 3 reads rows 6-7            │
│  (multiple readers = always safe)   │
└─────────────────────────────────────┘

Generation N+1 (being built - nobody reads here yet):
┌─────────────────────────────────────┐
│  next[][]     WRITE ONLY            │
│  Thread 0 writes rows 0-1           │
│  Thread 1 writes rows 2-3           │
│  Thread 2 writes rows 4-5           │
│  Thread 3 writes rows 6-7           │
│  (each thread owns different rows)  │
└─────────────────────────────────────┘

After all threads finish:
  swap(current, next)   ← just swapping two pointers, O(1)
```

No atomics needed. No critical sections. No locks.  
The double buffer design **eliminates** the race condition structurally.